# Notebook 1 — Introduzione a Google Cloud Vision API

**Corso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lezione 1:** Introduzione

Questo notebook accompagna il manuale *Google Vision API for Dummies*. Qui troverai:

- Spiegazioni brevi di ogni concetto (già sviluppato approfonditamente nel manuale)
- Celle di codice con esempi eseguibili usando l'API reale di Google Vision
- Esercizi con celle che completi tu, seguiti da una soluzione espandibile (fai clic su **Vedi soluzione** per controllare la tua proposta)

> **Prima di eseguire il codice reale di questo notebook** hai bisogno di un progetto Google Cloud con Vision API abilitata e un file di credenziali JSON. Se non lo hai ancora, apri prima il notebook `02_Configuracion_y_Setup_IT.ipynb` e torna dopo — qui puoi continuare a leggere e studiare la teoria senza problemi.

## 1. Cos'è Google Cloud Vision API?

L'intelligenza artificiale (IA) consente alle macchine di svolgere compiti che una volta richiedevano intelligenza umana. La visione artificiale è una delle sue applicazioni più evidenti: la capacità di interpretare dati visuali. Google mette questa capacità a tua disposizione con la **Google Cloud Vision API**, un servizio cloud che analizza immagini usando modelli di machine learning già addestrati, senza che tu debba addestrare nulla da solo.

Invii un'immagine, ti autentichi con le tue credenziali, e ricevi una risposta strutturata in formato JSON (notazione di oggetti JavaScript).

**Cosa può rilevare l'API (lo vedrai lezione dopo lezione in questo corso):**

- **Etichette**: oggetti, scene e concetti ("cane", "albero", "auto")
- **Volti**: posizione di occhi, naso e bocca, e probabilità di emozioni (gioia, rabbia, sorpresa, tristezza)
- **Testo (OCR)**: riconoscimento ottico dei caratteri, stampato o manoscritto
- **Oggetti**: ubicazione precisa di ogni oggetto all'interno dell'immagine
- **Monumenti, loghi e contenuto esplicito**: funzionalità avanzate (lezione 6)

### L'esempio del robot Gopi

Il materiale originale del corso propone un caso molto illustrativo: un robot Raspberry Pi con fotocamera, chiamato *Gopi*. Gopi non esegue il modello di IA nel suo hardware — invia la foto che ha catturato al cloud, la Vision API l'analizza, e gli restituisce il risultato. Con il rilevamento di volti ed emozioni, Gopi può avvicinarsi a un volto sorridente o allontanarsi davanti a rabbia o sorpresa. Lo ricreerai nell'esercizio 2.

## 2. Vision API rispetto a AutoML Vision API

All'interno di "Vision AI", Google separa due servizi che è importante differenziare prima di scrivere codice:

| | **Vision API** | **AutoML Vision API** |
|---|---|---|
| Modello | Preaddestrato da Google | Addestri il tuo modello con le tue immagini |
| Interfaccia | Codice (API REST o client Python) | Interfaccia grafica per caricare e etichettare immagini |
| Quando sceglierla | Ti basta il riconoscimento generico già addestrato | Hai bisogno di categorie molto specifiche del tuo business |
| Costo | Pagamento al consumo | Costo aggiuntivo per addestramento |

In questo corso lavorerai sempre con la **Vision API** (il modello preaddestrato), attraverso la libreria client Python `google-cloud-vision`.

## 3. Preparare il client Python

La cella seguente crea il client che userai durante tutto il notebook. Se hai già configurato la variabile di ambiente `GOOGLE_APPLICATION_CREDENTIALS` (lo vedrai nel notebook 2), questa cella dovrebbe eseguirsi senza errori.

In [ ]:
import os

# Ajusta esta ruta a tu archivo de credenciales JSON (ver notebook 02)
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision

try:
    client = vision.ImageAnnotatorClient()
    print("Cliente creado correctamente.")
except Exception as e:
    print("No se pudo crear el cliente. Revisa tus credenciales:", e)


## Esercizio 1 — Rilevamento di etichette di base

Completa la funzione `detectar_etiquetas` in modo che:

1. Apra l'immagine indicata in `image_path` in modalità binaria
2. Crei un oggetto `vision.Image` con il suo contenuto
3. Chiami `client.label_detection(...)`
4. Restituisca un elenco di tuple `(descrizione, score)` ordinate da confidenza maggiore a minore

Usa qualsiasi tua immagine (ad esempio, una foto di un animale, un paesaggio o un oggetto dalla tua scrivania) salvata nella stessa cartella di questo notebook.

In [ ]:
def detectar_etiquetas(image_path):
    # TODO 1: abre el archivo en modo binario y lee su contenido
    content = ...

    # TODO 2: crea el objeto vision.Image
    image = ...

    # TODO 3: llama al método de detección de etiquetas
    response = ...

    # TODO 4: construye y devuelve la lista (descripcion, score) ordenada por score descendente
    etiquetas = ...
    return etiquetas


# Descomenta y ajusta la ruta cuando tengas una imagen propia:
# resultado = detectar_etiquetas("mi_imagen.jpg")
# for descripcion, score in resultado:
#     print(f"{descripcion}: {score:.2f}")


<details>
<summary><strong>Vedi soluzione</strong></summary>

```python
def detectar_etiquetas(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.label_detection(image=image)

    etiquetas = [(label.description, label.score) for label in response.label_annotations]
    etiquetas.sort(key=lambda x: x[1], reverse=True)
    return etiquetas


resultado = detectar_etiquetas("mia_immagine.jpg")
for descripcion, score in resultado:
    print(f"{descripcion}: {score:.2f}")
```

</details>

## Esercizio 2 — Simulazione del robot Gopi

Ricrea la logica decisionale di Gopi: data un'immagine con un volto, il robot deve decidere se "avvicinarsi" (se rileva gioia con probabilità alta) o "allontanarsi" (se rileva rabbia o sorpresa con probabilità alta).

Suggerimenti:

- Usa `client.face_detection(image=image)` e prendi il primo volto da `response.face_annotations`
- I campi `joy_likelihood`, `anger_likelihood` e `surprise_likelihood` restituiscono un intero che corrisponde all'enumerazione `Likelihood`: `UNKNOWN` (0), `VERY_UNLIKELY` (1), `UNLIKELY` (2), `POSSIBLE` (3), `LIKELY` (4), `VERY_LIKELY` (5)
- Considera "probabilità alta" qualsiasi valore uguale o superiore a `LIKELY` (4)

In [ ]:
def decision_gopi(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.face_detection(image=image)

    if not response.face_annotations:
        return "No se detectó ningún rostro."

    rostro = response.face_annotations[0]

    # TODO 1: define el umbral "alto" (pista: el valor entero de LIKELY)
    umbral_alto = ...

    # TODO 2: decide la acción de Gopi según joy_likelihood, anger_likelihood y surprise_likelihood
    if ...:
        return "Gopi se acerca (detectó alegría)"
    elif ...:
        return "Gopi se aleja (detectó ira o sorpresa)"
    else:
        return "Gopi se queda quieto (emoción no concluyente)"


# print(decision_gopi("cara.jpg"))


<details>
<summary><strong>Vedi soluzione</strong></summary>

```python
def decision_gopi(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.face_detection(image=image)

    if not response.face_annotations:
        return "Nessun volto rilevato."

    rostro = response.face_annotations[0]
    umbral_alto = 4  # vision.Likelihood.LIKELY

    if rostro.joy_likelihood >= umbral_alto:
        return "Gopi si avvicina (ha rilevato gioia)"
    elif rostro.anger_likelihood >= umbral_alto or rostro.surprise_likelihood >= umbral_alto:
        return "Gopi si allontana (ha rilevato rabbia o sorpresa)"
    else:
        return "Gopi sta fermo (emozione non conclusiva)"


print(decision_gopi("viso.jpg"))
```

</details>

## Esercizio 3 — Esplora il client

Con il client già creato, esegui `dir(client)` e annota (nella cella Markdown seguente, modificandola tu) almeno cinque metodi che riconosci o che ti colpiscono in vista delle prossime lezioni.

In [ ]:
# Ejecuta esta celda y explora el resultado
print([m for m in dir(client) if not m.startswith("_")])


**Le tue note** (modifica questa cella):

1. `label_detection` — ...
2. `text_detection` — ...
3. `face_detection` — ...
4. `safe_search_detection` — ...
5. `image_properties` — ...

---

**Prossimo passo:** apri `02_Configuracion_y_Setup_IT.ipynb` per configurare completamente il tuo ambiente, o `03_Texto_en_Imagenes_IT.ipynb` se è già pronto.